# Fall & Call: Emergency Keyword Spotting (KWS)
This notebook trains a CNN model to recognize "HELP", "EMERGENCY", "CANCEL" and "BACKGROUND NOISE".

### 1. Setup and Hyperparameters

In [ ]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix

# MFCC Parameters (Matching Lab 4)
N_MFCC = 13
SAMPLING_RATE = 16000
FRAME_SIZE = 512
HOP_LENGTH = 256
CLASSES = ['help', 'emergency', 'cancel', 'background']

### 2. Model Training
We use the same CNN architecture (Conv2D -> MaxPooling -> Depthwise -> Dense) from the lab.

In [ ]:
# (Assuming train_mfccs, train_labels, val_mfccs, val_labels are prepared)
model = models.Sequential([
    layers.Input(shape=(64, 13, 1)),
    layers.Conv2D(16, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.DepthwiseConv2D((3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(32, activation='relu'),
    layers.Dense(len(CLASSES), activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model.fit(train_mfccs, train_labels, validation_data=(val_mfccs, val_labels), epochs=50)

### 3. Evaluating the Model (Following Lab Structure)
This section generates the graphs you need for your report's "Evaluation" section.

In [ ]:
# Plot Training & Validation Accuracy/Loss
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend()
plt.show()

### 4. Confusion Matrix
Used to identify which keywords are being confused (e.g., if 'Help' sounds too much like 'Emergency').

In [ ]:
y_pred = np.argmax(model.predict(test_mfccs), axis=1)
cm = confusion_matrix(test_labels, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASSES, yticklabels=CLASSES, cmap='Purples')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()